# Agentic RAG over PDFs — ingestion → hybrid retrieval → grounded answers

A complete, Colab-friendly pipeline that:

1. **Ingests** a PDF (PyMuPDF native text → PP-OCRv6 only when a page really needs OCR → Docling for headings/tables/figures)
2. **Indexes** it with FAISS (dense) + BM25S (keyword), both with metadata filtering
3. **Answers** questions through an *agent controller* that picks the retrieval strategy, a cross-encoder reranker, an evidence-sufficiency checker (one retry allowed), and a grounded answer with **page-level citations**

```
User query
   │
   ▼
Agent controller (Qwen3-4B) ── picks one of:
   │     direct │ rewrite │ multi-query │ metadata filter
   ▼
Hybrid retrieval  =  FAISS (e5-small)  +  BM25S  +  metadata filters  →  RRF fusion
   ▼
Cross-encoder reranker (MiniLM)
   ▼
Evidence sufficiency checker ──(not enough?)──► ONE extra retrieval attempt
   ▼
Grounded answer generation (Qwen3) with [p.N] citations
```

**LLM:** runs **locally** — `Qwen3-4B-Q4_K_M.gguf` (~2.5 GB) through **llama.cpp**. It is downloaded **once** and cached on your Google Drive, so later sessions skip the download. Use a **GPU runtime** (Runtime → Change runtime type → T4); on CPU it works but is slow. Prefer an API instead? Set `llm_backend = "groq"` in the config cell.

**Tokens:** add `HF_TOKEN` as a Colab *Secret* (🔑 icon in the left sidebar, enable notebook access), or paste it when prompted. `GROQ_API_KEY` is only asked for if you switch to the Groq backend.

## 1. Install & configure

In [1]:
# ---- Install everything in one go (takes ~3-5 minutes on a fresh Colab) ----
!pip -q install pymupdf rapidocr onnxruntime docling \
                sentence-transformers faiss-cpu bm25s groq pillow numpy huggingface_hub

# llama.cpp Python bindings: pick the prebuilt CUDA wheel if a GPU is present, otherwise the CPU wheel.
!nvidia-smi -L > /dev/null 2>&1 && pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 || pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
# ---- Secrets, model names and every tunable knob live here ----
import os, re, json, time, math, getpass, zipfile
from pathlib import Path
from dataclasses import dataclass
import numpy as np


def get_secret(name):
    # Look in Colab Secrets first, then environment variables, then ask nicely.
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name) or getpass.getpass(f"Enter {name}: ")


HF_TOKEN = get_secret("HF_TOKEN")
# Hugging Face libraries pick the token up from these env vars automatically.
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN


@dataclass
class Config:
    # --- models ---------------------------------------------------------
    llm_backend: str = "local"                               # "local" = Qwen3-4B GGUF via llama.cpp | "groq" = API
    gguf_repo: str = "Qwen/Qwen3-4B-GGUF"
    gguf_file: str = "Qwen3-4B-Q4_K_M.gguf"                  # ~2.5 GB, downloaded once
    use_drive: bool = True                                   # cache the GGUF on Google Drive so it's downloaded only once
    gguf_dir: str = "/content/models"                        # used when Drive is off / unavailable
    n_ctx: int = 8192                                        # context window for the local model
    groq_model: str = "qwen/qwen3.6-27b"                     # only used when llm_backend == "groq"
    groq_fallback_model: str = "openai/gpt-oss-120b"         # only used when llm_backend == "groq"
    embed_model: str = "intfloat/multilingual-e5-small"      # lightweight multilingual embeddings
    rerank_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    # ^ English-only. For multilingual PDFs use "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1".

    # --- ingestion ------------------------------------------------------
    min_native_chars: int = 40        # fewer characters than this -> page looks scanned -> OCR it
    ocr_zoom: float = 2.0             # render scale for OCR (2.0 ~ 144 DPI)
    ocr_conf_fallback: float = 0.85   # small-OCR confidence below this -> retry with the medium model
    use_docling: bool = True          # headings / tables / figures / reading order

    # --- chunking -------------------------------------------------------
    parent_chars: int = 1500          # parent = a section-sized block, never crosses a page
    child_chars: int = 450            # child = what actually gets embedded and searched
    child_overlap: int = 60

    # --- retrieval ------------------------------------------------------
    candidates_per_query: int = 20    # how many hits each retriever contributes per query
    rrf_k: int = 60                   # reciprocal-rank-fusion constant
    pool_size: int = 30               # candidates sent to the reranker
    top_n: int = 6                    # passages kept after reranking
    context_chars: int = 7000         # hard cap on evidence given to the LLM
    max_extra_retrievals: int = 1     # sufficiency checker may trigger this many extra rounds


CFG = Config()

WORK = Path("/content/agentic_rag")
ART = WORK / "artifacts"
ART.mkdir(parents=True, exist_ok=True)
print("✅ Config ready. LLM backend:", CFG.llm_backend)

Enter HF_TOKEN: ··········
✅ Config ready. LLM backend: local


## 2. Ingestion pipeline

Per page the logic is:

| Step | Tool | Why |
|---|---|---|
| Native text | **PyMuPDF** | Instant, and perfect when the PDF has a real text layer |
| Is it garbage / empty? | simple checks | Decide if OCR is *actually* needed |
| OCR | **PP-OCRv6 small** (ONNX Runtime) | Fast on CPU |
| Low OCR confidence? | **PP-OCRv6 medium** | Slower but more accurate, only when needed |
| Layout | **Docling** | Headings, tables (as Markdown), figures, reading order |

The output files use the same names and keys as your Phase 1 evaluation notebook
(`page_records.jsonl`, `child_chunks.jsonl`, `ocr_page_diagnostics.jsonl`), so that notebook still works on them.

> **Auto-download:** set `PAPER` in the next cell to an arXiv id/URL or a direct PDF link. Leave it empty to upload manually.


In [3]:
# ---- Get the PDF: auto-download from arXiv / a URL (or fall back to manual upload) ----
import requests

PAPER = "2008.07779"   # arXiv id (2008.07779 / 2008.07779v1), arXiv URL, or any direct .pdf URL.
                       # Set to "" to fall back to manual upload.

def fetch_paper(src, dest_dir):
    src = src.strip()
    m = re.search(r"(\d{4}\.\d{4,5}(?:v\d+)?)", src) if "arxiv" in src.lower() or re.fullmatch(r"[\d.v]+", src) else None
    url = f"https://arxiv.org/pdf/{m.group(1)}" if m else src
    print(f"⬇️ Downloading {url}")
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60, allow_redirects=True)
    r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        raise ValueError("Downloaded file is not a PDF")
    name = (m.group(1) if m else Path(url.split("?")[0]).stem or "paper") + ".pdf"
    path = Path(dest_dir) / name
    path.write_bytes(r.content)
    return path

PDF_PATH = ""
if PAPER:
    try:
        PDF_PATH = fetch_paper(PAPER, WORK)
    except Exception as e:
        print(f"⚠️ Auto-download failed ({type(e).__name__}: {e}) - falling back to manual upload.")

if not PDF_PATH:
    from google.colab import files
    print("📂 Upload the PDF you want to chat with:")
    up = files.upload()
    name = next(n for n in up if n.lower().endswith(".pdf"))
    PDF_PATH = WORK / name
    PDF_PATH.write_bytes(up[name])

PDF_PATH = Path(PDF_PATH)
DOC_NAME = PDF_PATH.name
print(f"✅ Using {DOC_NAME} ({PDF_PATH.stat().st_size / 1e6:.2f} MB)")


⬇️ Downloading https://arxiv.org/pdf/2008.07779
✅ Using 2008.07779.pdf (0.61 MB)


In [4]:
# ---- OCR helpers: PP-OCRv6 via ONNX Runtime (RapidOCR wrapper) ----
import pymupdf as fitz

_OCR_ENGINES = {}   # engines are heavy, so we build each tier once and reuse it


def get_ocr_engine(tier):
    # tier = "small" (fast, default) or "medium" (accurate fallback)
    if tier in _OCR_ENGINES:
        return _OCR_ENGINES[tier]
    from rapidocr import RapidOCR, OCRVersion, ModelType
    try:
        wanted = ModelType.SMALL if tier == "small" else ModelType.MEDIUM
        engine = RapidOCR(params={
            "Det.ocr_version": OCRVersion.PPOCRV6, "Det.model_type": wanted,
            "Rec.ocr_version": OCRVersion.PPOCRV6, "Rec.model_type": wanted,
        })
        label = f"PP-OCRv6-{tier}"
    except Exception as e:
        # If v6 weights can't be fetched for some reason, degrade to v5 instead of crashing.
        print(f"⚠️ PP-OCRv6 {tier} unavailable ({type(e).__name__}); falling back to PP-OCRv5.")
        wanted = ModelType.MOBILE if tier == "small" else ModelType.SERVER
        engine = RapidOCR(params={
            "Det.ocr_version": OCRVersion.PPOCRV5, "Det.model_type": wanted,
            "Rec.ocr_version": OCRVersion.PPOCRV5, "Rec.model_type": wanted,
        })
        label = f"PP-OCRv5-{'mobile' if tier == 'small' else 'server'}"
    _OCR_ENGINES[tier] = (engine, label)
    return _OCR_ENGINES[tier]


def render_page(page, zoom):
    # Rasterise a PDF page into an RGB numpy array.
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
    return np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)


def ocr_image(img, tier):
    # Run OCR and rebuild reading order: group boxes into lines (top->bottom), then left->right.
    # Returns (text, mean_confidence, engine_label).
    engine, label = get_ocr_engine(tier)
    out = engine(img)
    if out is None or out.txts is None or len(out.txts) == 0:
        return "", 0.0, label

    boxes = np.array(out.boxes, dtype=float)              # (n, 4, 2) corner points
    txts, scores = list(out.txts), [float(s) for s in out.scores]
    y_center = boxes[:, :, 1].mean(axis=1)
    height = np.median(boxes[:, :, 1].max(axis=1) - boxes[:, :, 1].min(axis=1)) or 1.0

    lines, current = [], []
    for i in np.argsort(y_center):
        if current and abs(y_center[i] - np.mean(y_center[current])) > 0.6 * height:
            lines.append(current)
            current = []
        current.append(i)
    if current:
        lines.append(current)

    text = "\n".join(
        " ".join(txts[j] for j in sorted(line, key=lambda j: boxes[j, :, 0].min()))
        for line in lines
    )
    return text, float(np.mean(scores)), label


def native_text_is_usable(text):
    # True if the PDF's own text layer looks real; False means "scanned or garbled -> OCR it".
    if len(text) < CFG.min_native_chars:
        return False
    sane = sum(ch.isalnum() or ch.isspace() or ch in ".,;:!?()-_/'\"%&@#+=*[]{}<>|" for ch in text)
    return (sane / len(text)) > 0.80 and "\ufffd" not in text[:2000]

In [5]:
# ---- Pass 1: native text, OCR where needed -> page_records + diagnostics ----
def extract_pages(pdf_path):
    doc = fitz.open(pdf_path)
    page_records, diagnostics = [], []

    for idx in range(len(doc)):
        page_no = idx + 1
        page = doc[idx]
        t0 = time.time()

        native = page.get_text("text").strip()
        rec = {"page_number": page_no, "document": DOC_NAME, "native_chars": len(native)}
        diag = {"page_number": page_no, "native_chars": len(native)}

        if native_text_is_usable(native):
            # Cheap path: the PDF already contains good text, so no OCR at all.
            rec.update(text=native, ocr_used=False, source="pymupdf", ocr_engine=None, ocr_confidence=None)
            diag.update(ocr_used=False, reason="usable native text")
        else:
            img = render_page(page, CFG.ocr_zoom)
            text, conf, engine = ocr_image(img, "small")
            fallback = False
            if conf < CFG.ocr_conf_fallback:
                # Small model wasn't confident -> try the medium model and keep the better result.
                text2, conf2, engine2 = ocr_image(img, "medium")
                fallback = True
                if conf2 >= conf and text2.strip():
                    text, conf, engine = text2, conf2, engine2
            # If OCR found nothing but there was a little native text, keep the native text.
            final = text if text.strip() else native
            rec.update(text=final, ocr_text=text, ocr_used=True, source="ocr",
                       ocr_engine=engine, ocr_confidence=round(conf, 4))
            diag.update(ocr_used=True, engine=engine, ocr_confidence=round(conf, 4),
                        medium_fallback_tried=fallback, ocr_chars=len(text),
                        reason="no/garbled native text")

        diag["seconds"] = round(time.time() - t0, 3)
        page_records.append(rec)
        diagnostics.append(diag)
        print(f"  page {page_no:>3}/{len(doc)}  {rec['source']:<8} chars={len(rec['text'])}")

    doc.close()
    return page_records, diagnostics


print("📄 Extracting pages ...")
page_records, ocr_diags = extract_pages(PDF_PATH)
n_ocr = sum(r["ocr_used"] for r in page_records)
print(f"✅ {len(page_records)} pages | {n_ocr} needed OCR | {len(page_records) - n_ocr} used native text")

📄 Extracting pages ...
  page   1/6  pymupdf  chars=2850
  page   2/6  pymupdf  chars=3953
  page   3/6  pymupdf  chars=1438
  page   4/6  pymupdf  chars=2680
  page   5/6  pymupdf  chars=2154
  page   6/6  pymupdf  chars=2922
✅ 6 pages | 0 needed OCR | 6 used native text


In [6]:
# ---- Pass 2: Docling layout (headings, tables, figures, reading order) ----
# We turn Docling's output into simple "blocks" per page:
#   {"type": "heading" | "text" | "table" | "figure", "text": ..., "level": int}
# Docling's own OCR is switched off - we already handled OCR above with PP-OCRv6.

def run_docling(pdf_path):
    from docling.document_converter import DocumentConverter, PdfFormatOption
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions

    opts = PdfPipelineOptions()
    opts.do_ocr = False
    opts.do_table_structure = True
    converter = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)})
    result = converter.convert(str(pdf_path))
    ddoc = result.document

    blocks_by_page = {}
    for item, level in ddoc.iterate_items():
        if not getattr(item, "prov", None):
            continue
        page_no = item.prov[0].page_no
        label = getattr(item, "label", "")
        label = getattr(label, "value", str(label))           # enum -> "section_header", "table", ...

        if label in ("page_header", "page_footer"):
            continue                                           # running headers/footers are noise
        if label in ("section_header", "title"):
            btype, text = "heading", (item.text or "").strip()
        elif label == "table":
            btype = "table"
            try:
                text = item.export_to_markdown(doc=ddoc)
            except TypeError:                                  # older Docling versions
                text = item.export_to_markdown()
        elif label in ("picture", "figure", "chart"):
            btype, text = "figure", ""
        else:
            btype, text = "text", (getattr(item, "text", "") or "").strip()

        if btype != "figure" and not text:
            continue
        blocks_by_page.setdefault(page_no, []).append({
            "type": btype, "text": text,
            "level": int(getattr(item, "level", 1) or 1) if btype == "heading" else 0,
        })
    return blocks_by_page


docling_blocks = {}
if CFG.use_docling:
    try:
        t0 = time.time()
        print("🧱 Running Docling layout analysis (slowest step on CPU, be patient) ...")
        docling_blocks = run_docling(PDF_PATH)
        print(f"✅ Docling done in {time.time() - t0:.0f}s - layout found on {len(docling_blocks)} pages")
    except Exception as e:
        # Docling is a bonus, not a requirement: fall back to plain page text.
        print(f"⚠️ Docling failed ({type(e).__name__}: {e}). Continuing with plain-text blocks.")
else:
    print("Docling disabled in CFG - using plain-text blocks.")

🧱 Running Docling layout analysis (slowest step on CPU, be patient) ...


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

✅ Docling done in 63s - layout found on 6 pages


### Chunking

Each page becomes **parent chunks** (≈1500 chars, split on headings so a parent stays inside one section) and each parent is cut into small **child chunks** (≈450 chars) that get embedded and searched. The LLM later reads the *parent* of every winning child, so it sees enough surrounding context.

Parents never cross a page boundary, which is what keeps the **page-level citations** honest. Tables are chunked row-wise with the header row repeated in every piece.

In [7]:
# ---- Chunking helpers ----
def split_text(text, size, overlap):
    # Sentence-aware packing with a little overlap between neighbouring chunks.
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    pieces = []
    for s in sentences:
        while len(s) > size:                         # a single huge "sentence" (e.g. a code line)
            cut = s.rfind(" ", 0, size)
            cut = cut if cut > size // 2 else size
            pieces.append(s[:cut].strip())
            s = s[cut:].strip()
        if s:
            pieces.append(s)

    chunks, cur, cur_len = [], [], 0
    for s in pieces:
        if cur and cur_len + len(s) + 1 > size:
            chunks.append(" ".join(cur))
            keep, kept_len = [], 0                    # carry the tail forward as overlap
            for t in reversed(cur):
                if kept_len + len(t) > overlap:
                    break
                keep.insert(0, t)
                kept_len += len(t) + 1
            cur, cur_len = keep, kept_len
        cur.append(s)
        cur_len += len(s) + 1
    if cur:
        chunks.append(" ".join(cur))
    return chunks or ([text] if text.strip() else [])


def split_table(md_table, size):
    # Keep table rows intact and repeat the header in each chunk so every piece is self-explanatory.
    lines = [l for l in md_table.split("\n") if l.strip()]
    if len("\n".join(lines)) <= size or len(lines) < 3:
        return ["\n".join(lines)]
    n_header = 2 if "---" in lines[1] else 1
    header, rows = lines[:n_header], lines[n_header:]
    chunks, cur = [], []
    for row in rows:
        if cur and len("\n".join(header + cur + [row])) > size:
            chunks.append("\n".join(header + cur))
            cur = []
        cur.append(row)
    if cur:
        chunks.append("\n".join(header + cur))
    return chunks


def looks_like_heading(line):
    # Cheap heading guess for pages Docling didn't cover (e.g. scanned pages).
    line = line.strip()
    if not (3 <= len(line) <= 80) or line.endswith((".", ",", ";")):
        return None
    numbered = re.match(r"^(\d+(\.\d+)*)[.)]?\s+[A-Z]", line)
    if numbered and len(line.split()) <= 7:        # short numbered line = likely a heading, not a list step
        return 1 + numbered.group(1).count(".")
    if line.isupper() and len(line.split()) <= 8:
        return 1
    return None


def get_page_blocks(rec):
    # Prefer Docling's structure for native-text pages; otherwise split the plain text ourselves,
    # line by line: a heading-looking line starts a new heading block, blank lines end paragraphs.
    blocks = docling_blocks.get(rec["page_number"])
    if blocks and not rec["ocr_used"] and any(b["type"] != "figure" for b in blocks):
        return blocks
    out, para = [], []

    def close_para():
        if para:
            out.append({"type": "text", "text": "\n".join(para), "level": 0})
            para.clear()

    for line in rec["text"].split("\n"):
        if not line.strip():
            close_para()
            continue
        lvl = looks_like_heading(line)
        if lvl:
            close_para()
            out.append({"type": "heading", "text": line.strip(), "level": lvl})
        else:
            para.append(line.strip())
    close_para()
    return out


def build_chunks(page_records):
    parents, children = [], []
    stack = {}                                       # heading level -> heading text (carried across pages)

    for rec in page_records:
        page_no = rec["page_number"]
        counter = {"n": 0}
        buf, buf_len = [], 0
        n_tables = n_figures = 0
        headings_on_page = []

        def emit(text, section, btype):
            # Create one parent chunk and all of its child chunks.
            counter["n"] += 1
            pid = f"p{page_no:03d}_{counter['n']:02d}"
            parents.append({"parent_id": pid, "text": text, "page_number": page_no, "section": section,
                            "block_type": btype, "document": DOC_NAME})
            pieces = split_table(text, CFG.child_chars) if btype == "table" \
                else split_text(text, CFG.child_chars, CFG.child_overlap)
            for j, piece in enumerate(pieces):
                children.append({
                    "chunk_id": f"{pid}_c{j:02d}", "parent_id": pid, "text": piece,
                    "metadata": {"page_number": page_no, "section": section, "block_type": btype,
                                 "source": rec["source"], "document": DOC_NAME},
                })

        def flush(section):
            nonlocal buf, buf_len
            if buf:
                joined = "\n".join(buf)
                if len(joined) > CFG.parent_chars * 1.3:     # one very long block: split it up
                    for part in split_text(joined, CFG.parent_chars, 0):
                        emit(part, section, "text")
                else:
                    emit(joined, section, "text")
            buf, buf_len = [], 0

        for b in get_page_blocks(rec):
            section = " > ".join(stack[k] for k in sorted(stack)) or "(no heading)"
            if b["type"] == "heading":
                flush(section)
                lvl = b["level"] or 1
                for k in [k for k in stack if k >= lvl]:
                    del stack[k]
                stack[lvl] = b["text"][:80]
                headings_on_page.append(b["text"][:80])
            elif b["type"] == "figure":
                n_figures += 1
            elif b["type"] == "table":
                flush(section)
                n_tables += 1
                emit(b["text"], section, "table")
            else:
                if buf and buf_len + len(b["text"]) > CFG.parent_chars:
                    flush(section)
                buf.append(b["text"])
                buf_len += len(b["text"]) + 1
        flush(" > ".join(stack[k] for k in sorted(stack)) or "(no heading)")

        rec.update(headings=headings_on_page, n_tables=n_tables, n_figures=n_figures)

    return parents, children


parents, children = build_chunks(page_records)
print(f"✅ {len(parents)} parent chunks, {len(children)} child chunks")
print("Example child:", json.dumps(children[min(3, len(children) - 1)], ensure_ascii=False)[:400])

✅ 30 parent chunks, 56 child chunks
Example child: {"chunk_id": "p001_04_c00", "parent_id": "p001_04", "text": "Techniques for making future predictions based upon the present and past data, has always been an area with direct application to various real life problems. We are discussing a similar problem in this paper. The problem statement is provided by Kaggle, which also serves as an ongoing competition on the Kaggle platform.", "metadata": {"p


In [8]:
# ---- Save artifacts (same file names as the Phase 1 evaluation notebook expects) ----
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl(ART / "page_records.jsonl", page_records)
write_jsonl(ART / "child_chunks.jsonl", children)
write_jsonl(ART / "parent_chunks.jsonl", parents)
write_jsonl(ART / "ocr_page_diagnostics.jsonl", ocr_diags)

(ART / "ingestion_summary.json").write_text(json.dumps({
    "document": DOC_NAME, "pages": len(page_records),
    "pages_ocr": sum(r["ocr_used"] for r in page_records),
    "parents": len(parents), "children": len(children),
    "docling_pages": len(docling_blocks),
}, indent=2), encoding="utf-8")

zip_path = WORK / "phase1_artifacts_agentic_rag.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ART.glob("*"):
        z.write(f, f"artifacts/{f.name}")
print(f"💾 Artifacts saved in {ART}  (zipped copy: {zip_path.name})")

💾 Artifacts saved in /content/agentic_rag/artifacts  (zipped copy: phase1_artifacts_agentic_rag.zip)


## 3. Hybrid retrieval engine

* **Dense**: `multilingual-e5-small` embeddings in a **FAISS** inner-product index (e5 expects `query:` / `passage:` prefixes — handled for you).
* **Sparse**: **BM25S** keyword scoring.
* **Metadata filters**: page list, page range, section substring, block type (e.g. only tables). Filters are applied *inside* both searches, not afterwards, so a filtered query never comes back empty just because the best hits were elsewhere.
* The two ranked lists are merged with **Reciprocal Rank Fusion** (no score-scale headaches).

In [9]:
# ---- Embeddings + FAISS + BM25S ----
import faiss, bm25s
from sentence_transformers import SentenceTransformer

class HybridRetriever:
    def __init__(self, children, parents):
        self.children = children
        self.parent_by_id = {p["parent_id"]: p for p in parents}
        self.n = len(children)

        # What we index = section path + chunk text, so headings help retrieval too.
        self.index_texts = [f"{c['metadata']['section']}\n{c['text']}" for c in children]

        # Metadata as numpy arrays -> filtering is just boolean masks.
        self.pages = np.array([c["metadata"]["page_number"] for c in children])
        self.sections = [c["metadata"]["section"].lower() for c in children]
        self.btypes = [c["metadata"]["block_type"] for c in children]

        # --- dense side ---
        print("🔢 Loading embedding model and encoding chunks ...")
        self.embedder = SentenceTransformer(CFG.embed_model)
        vecs = self.embedder.encode(
            ["passage: " + t for t in self.index_texts],
            batch_size=32, normalize_embeddings=True, show_progress_bar=True,
        ).astype("float32")
        self.faiss = faiss.IndexFlatIP(vecs.shape[1])      # cosine similarity (vectors are normalised)
        self.faiss.add(vecs)

        # --- sparse side ---
        tokens = bm25s.tokenize(self.index_texts, stopwords=None, show_progress=False)
        self.bm25 = bm25s.BM25()
        self.bm25.index(tokens, show_progress=False)

    # ---------- metadata filtering ----------
    def allowed_mask(self, filters):
        # Returns a boolean mask over all chunks (None = no filter).
        if not filters:
            return None
        mask = np.ones(self.n, dtype=bool)
        if filters.get("pages"):
            mask &= np.isin(self.pages, filters["pages"])
        if filters.get("page_range"):
            lo, hi = filters["page_range"]
            mask &= (self.pages >= lo) & (self.pages <= hi)
        if filters.get("section"):
            needle = filters["section"].lower()
            mask &= np.array([needle in s for s in self.sections])
        if filters.get("block_type"):
            mask &= np.array([b == filters["block_type"] for b in self.btypes])
        return mask

    # ---------- single-query search ----------
    def _dense(self, query, k, mask):
        qv = self.embedder.encode(["query: " + query], normalize_embeddings=True).astype("float32")
        if mask is None:
            _, ids = self.faiss.search(qv, k)
        else:
            allowed = np.where(mask)[0].astype("int64")
            if len(allowed) == 0:
                return []
            sel = faiss.IDSelectorBatch(allowed)           # FAISS only searches the allowed ids
            _, ids = self.faiss.search(qv, min(k, len(allowed)), params=faiss.SearchParameters(sel=sel))
        return [int(i) for i in ids[0] if i >= 0]

    def _sparse(self, query, k, mask):
        q_tokens = bm25s.tokenize(query, stopwords=None, show_progress=False, return_ids=False)[0]
        if not q_tokens:
            return []
        scores = np.array(self.bm25.get_scores(q_tokens))
        if mask is not None:
            scores = np.where(mask, scores, -1.0)
        order = np.argsort(-scores)[:k]
        return [int(i) for i in order if scores[i] > 0]    # score 0 = no keyword overlap, skip

    def search(self, query, k=None, filters=None):
        # Hybrid search for ONE query. Returns chunk ids ranked by fused score.
        return self.search_many([query], k, filters)

    def search_many(self, queries, k=None, filters=None):
        # Hybrid search for several queries; every ranked list is fused with RRF.
        k = k or CFG.candidates_per_query
        mask = self.allowed_mask(filters)
        fused = {}
        for q in queries:
            for ranked in (self._dense(q, k, mask), self._sparse(q, k, mask)):
                for rank, cid in enumerate(ranked):
                    fused[cid] = fused.get(cid, 0.0) + 1.0 / (CFG.rrf_k + rank + 1)
        ranked_ids = sorted(fused, key=fused.get, reverse=True)
        return [{"id": cid, "rrf": fused[cid]} for cid in ranked_ids]


retriever = HybridRetriever(children, parents)

# Table of contents the controller can use to build section filters.
def build_outline(page_records, max_items=60):
    seen, outline = set(), []
    for r in page_records:
        for h in r.get("headings", []):
            if h not in seen:
                seen.add(h)
                outline.append(f"p.{r['page_number']}: {h}")
    return outline[:max_items]

DOC_OUTLINE = build_outline(page_records)
N_PAGES = len(page_records)
print(f"✅ Retriever ready: {retriever.n} chunks indexed | outline entries: {len(DOC_OUTLINE)}")

🔢 Loading embedding model and encoding chunks ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


✅ Retriever ready: 56 chunks indexed | outline entries: 22


In [10]:
# ---- Cross-encoder reranker ----
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(CFG.rerank_model, max_length=512)

def rerank(queries, candidates, top_n=None):
    # Score every candidate against every query with the cross-encoder, then pick the top_n.
    # With one query this is a plain sort. With several queries (multi-query search) we
    # take winners round-robin from each query's own ranking so every concept gets coverage.
    top_n = top_n or CFG.top_n
    candidates = candidates[:CFG.pool_size]
    if not candidates:
        return []
    texts = [retriever.children[c["id"]]["text"] for c in candidates]

    per_query_scores = []
    for q in queries:
        scores = reranker.predict([(q, t) for t in texts], show_progress_bar=False)
        per_query_scores.append(np.array(scores, dtype=float))

    rankings = [list(np.argsort(-s)) for s in per_query_scores]
    chosen, seen = [], set()
    depth = 0
    while len(chosen) < min(top_n, len(candidates)) and depth < len(candidates):
        for qi, ranking in enumerate(rankings):
            idx = ranking[depth]
            if idx not in seen and len(chosen) < top_n:
                seen.add(idx)
                chosen.append({"id": candidates[idx]["id"], "rerank_score": float(per_query_scores[qi][idx])})
        depth += 1

    # Present the final list best-first by each chunk's strongest score.
    return sorted(chosen, key=lambda c: c["rerank_score"], reverse=True)

print("✅ Reranker ready:", CFG.rerank_model)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Reranker ready: cross-encoder/ms-marco-MiniLM-L-6-v2


## 4. The agent — controller, sufficiency checker, grounded generator

All three roles are played by the same Qwen3 model, each with its own small prompt. The controller and checker are asked for **JSON only**, and everything they return is validated before use — a hallucinated page number or an unknown section name can't break retrieval; it just gets dropped.

In [13]:
# ---- LLM backend: local Qwen3-4B (llama.cpp) by default, Groq optional ----
THINK_RE = re.compile(r"<think>.*?</think>", re.S)      # Qwen3 may emit <think>...</think>; we strip it


def parse_json(text):
    # Pull the first {...} object out of an LLM reply; return None if it isn't valid JSON.
    if not text:
        return None
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None


if CFG.llm_backend == "local":
    import subprocess
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama

    # 1) Where to keep the 2.5 GB file: Google Drive survives Colab restarts, local disk doesn't.
    model_dir = CFG.gguf_dir
    if CFG.use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            model_dir = "/content/drive/MyDrive/gguf_models"
        except Exception as e:
            print(f"(Drive unavailable: {type(e).__name__}) - caching on local disk instead.")
    Path(model_dir).mkdir(parents=True, exist_ok=True)
    gguf_path = Path(model_dir) / CFG.gguf_file

    def gguf_ok(p):
        # A healthy Q4_K_M file is ~2.5 GB and starts with the magic bytes "GGUF".
        # (The old ">1 GB" check let half-downloaded files through.)
        try:
            if not p.exists() or p.stat().st_size < 2.0e9:
                return False
            with open(p, "rb") as f:
                return f.read(4) == b"GGUF"
        except OSError:
            return False

    def download_model():
        print(f"⬇️ Downloading {CFG.gguf_file} (one time only) ...")
        return Path(hf_hub_download(repo_id=CFG.gguf_repo, filename=CFG.gguf_file,
                                    local_dir=model_dir, token=HF_TOKEN, force_download=True))

    # 2) Use the cached file only if it is complete and valid.
    if gguf_ok(gguf_path):
        print(f"♻️ Using cached model: {gguf_path}")
    else:
        if gguf_path.exists():
            print(f"⚠️ Cached model looks incomplete/corrupt ({gguf_path.stat().st_size / 1e9:.2f} GB) - re-downloading.")
            gguf_path.unlink()
        gguf_path = download_model()

    # 3) Load it. n_gpu_layers=-1 puts every layer on the GPU when there is one.
    has_gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True).returncode == 0

    def load_llama(path, gpu):
        return Llama(model_path=str(path), n_ctx=CFG.n_ctx, n_gpu_layers=-1 if gpu else 0,
                     n_threads=os.cpu_count(), verbose=False)

    local_llm = None
    try:
        local_llm = load_llama(gguf_path, has_gpu)
    except Exception as e1:
        print(f"⚠️ Model failed to load ({type(e1).__name__}: {e1})")
        if has_gpu:                                    # maybe the CUDA wheel is the problem -> try CPU
            try:
                print("   retrying on CPU ...")
                local_llm, has_gpu = load_llama(gguf_path, False), False
            except Exception:
                local_llm = None
        if local_llm is None:                          # still failing -> the file itself is probably bad
            print("   deleting the cached file and downloading a fresh copy ...")
            gguf_path.unlink(missing_ok=True)
            gguf_path = download_model()
            try:
                local_llm = load_llama(gguf_path, has_gpu)
            except Exception as e2:
                raise RuntimeError(
                    "Could not load the GGUF even after a fresh download. Your llama-cpp-python is probably too old "
                    "for Qwen3. Re-run the install cell with: !pip install -U --force-reinstall llama-cpp-python "
                    "(then Runtime -> Restart session), or set CFG.llm_backend = 'groq'.") from e2

    LLM_LABEL = f"{CFG.gguf_file} (llama.cpp, {'GPU' if has_gpu else 'CPU - expect slow runs'})"
    if not has_gpu:
        print("⚠️ No GPU detected - the benchmark will be slow. Runtime -> Change runtime type -> T4 GPU.")

    def llm(messages, *, temperature=0.1, max_tokens=900, json_mode=False):
        # "/no_think" on the last user turn switches off Qwen3's long reasoning (far faster, fine for this task).
        messages = [dict(m) for m in messages]
        messages[-1]["content"] += " /no_think"
        kwargs = dict(messages=messages, temperature=temperature, max_tokens=max_tokens)
        try:
            if json_mode:
                out = local_llm.create_chat_completion(**kwargs, response_format={"type": "json_object"})
            else:
                out = local_llm.create_chat_completion(**kwargs)
        except Exception:
            out = local_llm.create_chat_completion(**kwargs)       # JSON grammar failed -> rely on the prompt
        return THINK_RE.sub("", out["choices"][0]["message"]["content"] or "").strip()

else:
    # ---------------- Groq API backend ----------------
    from groq import Groq
    GROQ_API_KEY = get_secret("GROQ_API_KEY")
    groq_client = Groq(api_key=GROQ_API_KEY)
    LLM_LABEL = f"{CFG.groq_model} (Groq)"

    def llm(messages, *, temperature=0.1, max_tokens=900, json_mode=False):
        # +1500 headroom: reasoning models spend tokens "thinking" before the visible answer.
        kwargs = dict(model=CFG.groq_model, messages=messages, temperature=temperature, max_tokens=max_tokens + 1500)
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}
        extras = {"reasoning_format": "hidden"}
        for attempt in range(4):
            try:
                resp = groq_client.chat.completions.create(**kwargs, **extras)
                return THINK_RE.sub("", resp.choices[0].message.content or "").strip()
            except Exception as e:
                msg = str(e).lower()
                if "model_not_found" in msg or "does not exist" in msg:
                    if CFG.groq_model == CFG.groq_fallback_model:
                        raise
                    print(f"   ⚠️ {CFG.groq_model} unavailable on Groq - switching to {CFG.groq_fallback_model}")
                    CFG.groq_model = kwargs["model"] = CFG.groq_fallback_model
                    continue
                if extras and ("reasoning" in msg or isinstance(e, TypeError)):
                    extras = {}
                    continue
                if "429" in msg or "rate" in msg:
                    wait = 2 ** (attempt + 1)
                    print(f"   (Groq rate limit - waiting {wait}s)")
                    time.sleep(wait)
                    continue
                if json_mode and "json" in msg and "response_format" in kwargs:
                    kwargs.pop("response_format")
                    continue
                raise
        raise RuntimeError("Groq call failed after several retries")


print("LLM test ->", llm([{"role": "user", "content": "Reply with the single word: ready"}], max_tokens=20))
print("Using:", LLM_LABEL)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
♻️ Using cached model: /content/drive/MyDrive/gguf_models/Qwen3-4B-Q4_K_M.gguf
⚠️ Model failed to load (ValueError: Failed to load model from file: /content/drive/MyDrive/gguf_models/Qwen3-4B-Q4_K_M.gguf)
   retrying on CPU ...
⚠️ No GPU detected - the benchmark will be slow. Runtime -> Change runtime type -> T4 GPU.
LLM test -> ready
Using: Qwen3-4B-Q4_K_M.gguf (llama.cpp, CPU - expect slow runs)


In [14]:
# ---- Agent controller: decides HOW to retrieve ----
CONTROLLER_SYSTEM = """You are the retrieval controller of a document question-answering system.
Pick exactly ONE retrieval operation for the user's question and reply with JSON only.

Operations:
- "direct": a clear factual question. queries = [the question as-is].
- "rewrite": the question is vague, uses pronouns, or is badly phrased. queries = [ONE clearer, self-contained search query].
- "multi_query": the question needs several separate concepts or a comparison. queries = 2-4 short sub-queries, one per concept.
- "metadata_filter": the question names a page, page range, section/chapter, or asks about a table. Put the search text in queries (1 query) and set filters.

JSON schema:
{"operation": "direct|rewrite|multi_query|metadata_filter",
 "queries": ["..."],
 "filters": {"pages": [int], "page_range": [int, int], "section": "text", "block_type": "table"},
 "reason": "one short sentence"}
Only include the filter keys you actually need; use {} when there are none.
Use "section" only with words that really appear in the document outline provided."""


def sanitize_plan(plan, question):
    # Never trust LLM output blindly: clamp it to something safe and valid.
    plan = plan if isinstance(plan, dict) else {}
    op = plan.get("operation")
    if op not in {"direct", "rewrite", "multi_query", "metadata_filter"}:
        op = "direct"

    queries = [q.strip() for q in plan.get("queries", []) if isinstance(q, str) and q.strip()][:4]
    if not queries or op == "direct":
        queries = [question]
    if op in ("rewrite", "metadata_filter"):
        queries = queries[:1]
    if op == "multi_query" and len(queries) < 2:
        op, queries = "direct", [question]

    filters = {}
    raw = plan.get("filters") if op == "metadata_filter" and isinstance(plan.get("filters"), dict) else {}
    pages = [int(p) for p in raw.get("pages", []) if str(p).isdigit() and 1 <= int(p) <= N_PAGES]
    if pages:
        filters["pages"] = pages
    pr = raw.get("page_range")
    if isinstance(pr, list) and len(pr) == 2 and all(str(x).isdigit() for x in pr):
        lo, hi = sorted((max(1, int(pr[0])), min(N_PAGES, int(pr[1]))))
        filters["page_range"] = [lo, hi]
    if isinstance(raw.get("section"), str) and raw["section"].strip():
        filters["section"] = raw["section"].strip()
    if raw.get("block_type") in ("table", "text"):
        filters["block_type"] = raw["block_type"]

    if op == "metadata_filter" and not filters:
        op = "direct"                         # the "filter" turned out to be empty -> plain search
    return {"operation": op, "queries": queries, "filters": filters, "reason": str(plan.get("reason", ""))[:200]}


def controller_decide(question):
    outline = "\n".join(DOC_OUTLINE) if DOC_OUTLINE else "(no outline available)"
    user_msg = (f"Document has {N_PAGES} pages.\nDocument outline:\n{outline}\n\n"
                f"User question: {question}\n\nReturn the JSON plan.")
    raw = llm([{"role": "system", "content": CONTROLLER_SYSTEM},
               {"role": "user", "content": user_msg}], json_mode=True, max_tokens=400)
    return sanitize_plan(parse_json(raw), question)

In [15]:
# ---- Evidence sufficiency checker ----
CHECKER_SYSTEM = """You judge whether retrieved passages contain enough evidence to answer a question.
Reply with JSON only:
{"sufficient": true|false, "missing": "what is missing (empty if sufficient)", "follow_up_query": "a better search query to find the missing evidence (empty if sufficient)"}
Be strict: "sufficient" means the passages directly support a complete answer, not merely mention the topic."""


def build_context(top, max_chars=None):
    # Turn reranked child chunks into labelled evidence blocks.
    # We hand the LLM each chunk's PARENT (more context), but only once per parent.
    max_chars = max_chars or CFG.context_chars
    blocks, seen_parents, used = [], set(), 0
    for item in top:
        child = retriever.children[item["id"]]
        pid = child["parent_id"]
        if pid in seen_parents:
            continue
        seen_parents.add(pid)
        parent = retriever.parent_by_id[pid]
        body = parent["text"]
        header = f"[Page {parent['page_number']} | {parent['section']}]"
        if used + len(body) > max_chars and blocks:
            break
        blocks.append({"page": parent["page_number"], "section": parent["section"],
                       "header": header, "text": body, "parent_id": pid})
        used += len(body)
    return blocks


def check_sufficiency(question, blocks):
    if not blocks:
        return {"sufficient": False, "missing": "nothing was retrieved", "follow_up_query": question}
    evidence = "\n\n".join(f"{b['header']}\n{b['text'][:800]}" for b in blocks)
    raw = llm([{"role": "system", "content": CHECKER_SYSTEM},
               {"role": "user", "content": f"Question: {question}\n\nPassages:\n{evidence}\n\nReturn the JSON verdict."}],
              json_mode=True, max_tokens=300)
    verdict = parse_json(raw) or {}
    return {"sufficient": bool(verdict.get("sufficient", True)),     # if parsing fails, don't loop forever
            "missing": str(verdict.get("missing", ""))[:300],
            "follow_up_query": str(verdict.get("follow_up_query", "")).strip()}

In [16]:
# ---- Grounded answer generation with page-level citations ----
ANSWER_SYSTEM = """You answer questions about a PDF document using ONLY the evidence passages provided.
Rules:
1. Use only facts that appear in the passages. Never use outside knowledge.
2. After every claim, cite its page like [p.7]. Use one bracket per page, e.g. [p.7] [p.9]. Only cite pages shown in the passage headers.
3. If the passages do not contain the answer, say clearly that the document does not provide it, and say what is missing.
4. If the evidence is only partial, answer the part you can support and state what is not covered.
5. Be concise and well organised. Keep commands, code and numbers exactly as written in the passages."""


def generate_answer(question, blocks):
    if not blocks:
        return "I couldn't find anything relevant to that in the document."
    evidence = "\n\n".join(f"{b['header']}\n{b['text']}" for b in blocks)
    return llm([{"role": "system", "content": ANSWER_SYSTEM},
                {"role": "user", "content": f"Evidence passages:\n\n{evidence}\n\nQuestion: {question}"}],
               temperature=0.2, max_tokens=1200)


def verify_citations(answer, blocks):
    # Check every [p.N] in the answer against pages we actually supplied.
    supplied = {b["page"] for b in blocks}
    cited = {int(n) for n in re.findall(r"p\.?\s*(\d+)", " ".join(re.findall(r"\[[^\]]*\]", answer)))}
    return sorted(cited & supplied), sorted(cited - supplied)

In [17]:
# ---- The orchestrator: ties every stage together ----
class AgenticRAG:
    def ask(self, question, verbose=True):
        trace, t_start = [], time.time()
        log = lambda s: (trace.append(s), print(s) if verbose else None)

        # 1) Controller picks the retrieval strategy
        plan = controller_decide(question)
        log(f"🧭 Controller: {plan['operation']} | queries={plan['queries']} | filters={plan['filters'] or '-'}")
        if plan["reason"]:
            log(f"   why: {plan['reason']}")

        # 2) Hybrid retrieval (+ graceful fallback if a filter matches nothing)
        candidates = retriever.search_many(plan["queries"], filters=plan["filters"])
        if plan["filters"] and not candidates:
            log("   ⚠️ filters matched nothing - retrying without filters")
            candidates = retriever.search_many(plan["queries"])
        log(f"🔎 Hybrid retrieval: {len(candidates)} candidates")

        # 3) Rerank. Multi-query ranks per sub-query; everything else ranks on the (rewritten) query.
        rerank_queries = plan["queries"] if plan["operation"] != "direct" else [question]
        top = rerank(rerank_queries, candidates)
        blocks = build_context(top)
        log(f"🏅 Reranked -> {len(top)} passages from pages {sorted({b['page'] for b in blocks})}")

        # 4) Sufficiency check, with at most N extra retrieval rounds
        extra_used = 0
        while True:
            verdict = check_sufficiency(question, blocks)
            log(f"🧪 Sufficiency: {'enough' if verdict['sufficient'] else 'NOT enough'}"
                + (f" (missing: {verdict['missing']})" if not verdict["sufficient"] else ""))
            if verdict["sufficient"] or extra_used >= CFG.max_extra_retrievals:
                break
            extra_used += 1
            follow_up = verdict["follow_up_query"] or f"{question} {verdict['missing']}"
            log(f"🔁 Extra retrieval with: '{follow_up}' (filters dropped to widen the net)")
            more = retriever.search_many([follow_up])
            merged = {c["id"]: c for c in more}                   # follow-up hits first ...
            for c in candidates:                                  # ... then the original ones
                merged.setdefault(c["id"], c)
            candidates = list(merged.values())
            top = rerank([question, follow_up], candidates)
            blocks = build_context(top)
            log(f"🏅 Re-reranked -> pages {sorted({b['page'] for b in blocks})}")

        # 5) Grounded generation
        answer = generate_answer(question, blocks)
        valid, bogus = verify_citations(answer, blocks)
        if bogus:
            log(f"   ⚠️ answer cites pages that weren't in the evidence: {bogus}")

        sources = [{"page": b["page"], "section": b["section"], "snippet": b["text"][:160].replace("\n", " ")}
                   for b in blocks]
        result = {"question": question, "answer": answer, "cited_pages": valid, "sources": sources,
                  "plan": plan, "sufficient": verdict["sufficient"], "extra_retrievals": extra_used,
                  "seconds": round(time.time() - t_start, 2), "trace": trace}
        return result


def show(result):
    print("\n" + "=" * 70)
    print("Q:", result["question"])
    print("=" * 70)
    print(result["answer"])
    print("\n📚 Cited pages:", result["cited_pages"] or "none")
    print("📄 Evidence used:")
    for s in result["sources"]:
        print(f"   p.{s['page']:<4} {s['section'][:50]:<50} {s['snippet'][:70]}...")
    print(f"⏱️ {result['seconds']}s | extra retrievals: {result['extra_retrievals']}")


rag = AgenticRAG()
print("✅ Agentic RAG is ready - ask away in the next cell.")

✅ Agentic RAG is ready - ask away in the next cell.


In [18]:
# ---- Page-aware routing: fixes questions like "What does page 3 cover?" ----
# Why "What does page 3 cover?" failed:
#   the 4B controller chose "direct" instead of "metadata_filter", so page 3 was never
#   targeted and semantic search can't match a question about a *page number*.
# Fix: (1) deterministic rules parse page / page-range references (no LLM guessing),
#      (2) "what does page N cover" skips similarity search and reads the page itself.

NUM_WORDS = {w: i for i, w in enumerate(
    "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen "
    "sixteen seventeen eighteen nineteen twenty".split())}
_NUM = r"(\d+|" + "|".join(NUM_WORDS) + r")"
_N = lambda s: int(s) if s.isdigit() else NUM_WORDS[s.lower()]

PAGE_RE = re.compile(
    rf"(?i)\b(?:pages?|pp?\.?)\s*{_NUM}(?:\s*(?:-|–|—|to|through|and|&)\s*{_NUM})?\b")
ORDINAL_RE = re.compile(r"(?i)\b(\d+)(?:st|nd|rd|th)\s+page\b")
OVERVIEW_RE = re.compile(
    r"(?i)\b(cover|covers|covered|contain|contains|about|summar\w*|overview|discuss\w*|"
    r"content|contents|talk\w*|describe\w*|explain\w*|what('s| is| does| do)?)\b")
LAST_PAGE_RE = re.compile(r"(?i)\b(last|final)\s+page\b")
FIRST_PAGE_RE = re.compile(r"(?i)\b(first)\s+page\b")


def detect_pages(question):
    pages, span = set(), []
    for m in PAGE_RE.finditer(question):
        a = _N(m.group(1))
        b = _N(m.group(2)) if m.group(2) else None
        sep = question[m.end(1):m.start(2)] if b else ""
        if b and re.search(r"(?i)\band\b|&", sep):
            pages.update([a, b])
        elif b:
            pages.update(range(min(a, b), max(a, b) + 1))
        else:
            pages.add(a)
        span.append(m.span())
    for m in ORDINAL_RE.finditer(question):
        pages.add(int(m.group(1))); span.append(m.span())
    if LAST_PAGE_RE.search(question):
        pages.add(N_PAGES)
    if FIRST_PAGE_RE.search(question):
        pages.add(1)
    stripped = question
    for s, e in sorted(span, reverse=True):
        stripped = stripped[:s] + " " + stripped[e:]
    return sorted(pages), re.sub(r"\s+", " ", stripped).strip(" ?.")


def is_page_overview(question, pages):
    # Question is about the page itself ("what does page 3 cover"), not a fact that happens to be on it.
    if not pages:
        return False
    _, rest = detect_pages(question)
    rest_words = [w for w in re.findall(r"[a-zA-Z]+", rest.lower())
                  if w not in {"what", "does", "do", "is", "are", "the", "on", "in", "of", "this", "about",
                               "cover", "covers", "contain", "contains", "talk", "discuss", "describe",
                               "summarize", "summary", "overview", "content", "contents", "document",
                               "paper", "give", "me", "a", "an", "tell", "explain", "and", "to", "show",
                               "page", "pages", "last", "first", "final", "p", "pp"}]
    return bool(OVERVIEW_RE.search(question)) and len(rest_words) <= 1


def rule_plan(question):
    # Deterministic router. Returns a plan, or None to let the LLM controller decide.
    pages, rest = detect_pages(question)
    if not pages:
        return None
    valid = [p for p in pages if 1 <= p <= N_PAGES]
    if not valid:
        return None
    return {"operation": "metadata_filter", "queries": [rest or question],
            "filters": {"pages": valid},
            "reason": f"rule-based: question references page(s) {pages}",
            "_requested_pages": pages}


if "_llm_controller_decide" not in globals():
    _llm_controller_decide = controller_decide     # keep the original LLM controller
def controller_decide(question):                       # same name -> AgenticRAG.ask picks it up automatically
    return rule_plan(question) or _llm_controller_decide(question)


def page_blocks(pages):
    # Evidence = the pages themselves, in reading order (no similarity search needed).
    blocks, used = [], 0
    for p in parents:
        if p["page_number"] in pages:
            if used + len(p["text"]) > CFG.context_chars and blocks:
                break
            blocks.append({"page": p["page_number"], "section": p["section"],
                           "header": f"[Page {p['page_number']} | {p['section']}]",
                           "text": p["text"], "parent_id": p["parent_id"]})
            used += len(p["text"])
    return blocks


class AgenticRAGv2(AgenticRAG):
    def ask(self, question, verbose=True):
        pages, _ = detect_pages(question)
        if not is_page_overview(question, pages):
            return super().ask(question, verbose)       # everything else: normal agent path

        t0, trace = time.time(), []
        log = lambda s: (trace.append(s), print(s) if verbose else None)
        out_of_range = [p for p in pages if not 1 <= p <= N_PAGES]
        valid = [p for p in pages if 1 <= p <= N_PAGES]
        log(f"🧭 Rule router: page_overview | pages={valid}")
        plan = {"operation": "page_overview", "queries": [question], "filters": {"pages": valid},
                "reason": "question asks what a specific page contains"}

        if not valid:
            ans = f"The document only has {N_PAGES} pages, so page {out_of_range} does not exist."
            return {"question": question, "answer": ans, "cited_pages": [], "sources": [], "plan": plan,
                    "sufficient": True, "extra_retrievals": 0, "seconds": round(time.time() - t0, 2), "trace": trace}

        blocks = page_blocks(set(valid))
        # structural facts from ingestion (tables / figures / headings) help the summary
        meta = []
        for r in page_records:
            if r["page_number"] in valid:
                meta.append(f"Page {r['page_number']}: headings={r.get('headings') or 'none'}, "
                            f"tables={r.get('n_tables', 0)}, figures={r.get('n_figures', 0)}")
        evidence = "\n\n".join(f"{b['header']}\n{b['text']}" for b in blocks)
        q = (f"Describe what {'page ' + str(valid[0]) if len(valid) == 1 else 'pages ' + str(valid)} cover: "
             f"the sections, main points, and any tables or figures. "
             f"Page structure info: {' | '.join(meta)}")
        answer = llm([{"role": "system", "content": ANSWER_SYSTEM},
                      {"role": "user", "content": f"Evidence passages:\n\n{evidence}\n\nQuestion: {q}"}],
                     temperature=0.2, max_tokens=900)
        if out_of_range:
            answer += f"\n\n(Note: the document only has {N_PAGES} pages; page(s) {out_of_range} do not exist.)"
        cited, bogus = verify_citations(answer, blocks)
        sources = [{"page": b["page"], "section": b["section"], "snippet": b["text"][:160].replace("\n", " ")} for b in blocks]
        log(f"📄 Read {len(blocks)} blocks directly from page(s) {valid}")
        return {"question": question, "answer": answer, "cited_pages": cited, "sources": sources, "plan": plan,
                "sufficient": True, "extra_retrievals": 0, "seconds": round(time.time() - t0, 2), "trace": trace}


rag = AgenticRAGv2()

# quick tests - no LLM needed for the routing part
for q in ["What does page 3 cover?", "Summarize pages 2-4", "what is on the last page",
          "What RMSE does XGBoost get on page 6?", "What does page 99 contain?"]:
    pg, rest = detect_pages(q)
    print(f"{q!r:55} pages={pg} overview={is_page_overview(q, pg)} rest={rest!r}")

show(rag.ask("What does page 3 cover?"))


'What does page 3 cover?'                               pages=[3] overview=True rest='What does cover'
'Summarize pages 2-4'                                   pages=[2, 3, 4] overview=True rest='Summarize'
'what is on the last page'                              pages=[6] overview=True rest='what is on the last page'
'What RMSE does XGBoost get on page 6?'                 pages=[6] overview=False rest='What RMSE does XGBoost get on'
'What does page 99 contain?'                            pages=[99] overview=True rest='What does contain'
🧭 Rule router: page_overview | pages=[3]
📄 Read 3 blocks directly from page(s) [3]

Q: What does page 3 cover?
Page 3 covers the following sections, main points, and any tables or figures:

- **Sections**: 
  - 3.2 Exploratory Data Analysis (EDA)
  - 3.3 Feature engineering

- **Main Points**:
  - **3.2 Exploratory Data Analysis (EDA)**: EDA was a required step to get insights for data preparation and feature extraction. Key observations included that th

## 5. Try it

Edit the questions below. They're written to exercise each controller path: a plain fact, a vague question (rewrite), a comparison (multi-query), and a page/table-specific one (metadata filter).

In [19]:
QUESTIONS = [
    "What is this document about?",                                   # direct / rewrite
    "Compare the two main approaches described in the document.",     # multi-query
    "What does page 3 cover?",                                        # metadata filter
    "Is there a table in the document? What does it contain?",        # metadata filter (tables)
]

for q in QUESTIONS:
    show(rag.ask(q, verbose=True))

🧭 Controller: direct | queries=['What is this document about?'] | filters=-
🔎 Hybrid retrieval: 36 candidates
🏅 Reranked -> 6 passages from pages [1, 2, 6]
🧪 Sufficiency: enough
   ⚠️ answer cites pages that weren't in the evidence: [4]

Q: What is this document about?
This document is about a project aimed at forecasting future sales using a time-series dataset provided by Kaggle. The project involves predicting the total sales of every product and store combination for the next month based on past data. The authors of the document have explored various forecasting techniques, including XGBoost, ARIMA, and LSTM, and have evaluated their performance using the Root Mean Squared Error (RMSE) metric. The document also discusses the importance of feature engineering and hyperparameter tuning in achieving accurate sales forecasts. [p.1] [p.2] [p.4] [p.6]

📚 Cited pages: [1, 2, 6]
📄 Evidence used:
   p.1    1 Introduction                                     Accurate sales forecasting is of u

In [ ]:
# ---- Interactive loop (type 'quit' to stop) ----
while True:
    q = input("\nAsk a question about the PDF (or 'quit'): ").strip()
    if not q or q.lower() in {"quit", "exit", "q"}:
        break
    show(rag.ask(q, verbose=True))

In [ ]:
# ---- Optional: download the ingestion artifacts (works with the Phase 1 evaluation notebook) ----
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print(f"(Not in Colab or download blocked: {e}) - the ZIP is at {zip_path}")

In [ ]:
# =====================================================================================
#  BENCHMARK CELL - upload PDF(s) + ground-truth JSON, ingest, run every question, score
# =====================================================================================
# What it measures, per question:
#   retrieval_hit / retrieval_recall : did the evidence given to the LLM include the gold page(s)?
#   citation_hit / citation_precision: did the answer cite the gold page(s)?
#   judge_score                      : LLM judge vs. ground-truth answer (1 / 0.5 / 0)
#   token_f1                         : cheap word-overlap score (no LLM, useful sanity check)
# Ground-truth JSON format: a list of {id, question, ground_truth_answer, evidence_location, category}
import pandas as pd, string
from collections import Counter

# ---------------- knobs ----------------
TARGET_PDF_HINT = ""        # if you upload several PDFs: part of the filename the JSON is about
USE_LLM_JUDGE = True        # False = skip the judge (faster, token-F1 only)
MAX_QUESTIONS = None        # e.g. 10 for a quick smoke test; None = all
PAUSE_SECONDS = 0.0         # raise (e.g. 1.0) if Groq rate-limits you
MANUAL_PDF_PATHS = []       # optional: ["/content/x.pdf"] to skip the upload dialog
MANUAL_JSON_PATH = ""       # optional: "/content/bench.json"

bench_dir = WORK / "benchmark"
bench_dir.mkdir(parents=True, exist_ok=True)

# ---------------- 1) upload ----------------
if MANUAL_PDF_PATHS and MANUAL_JSON_PATH:
    pdf_paths, json_path = [Path(p) for p in MANUAL_PDF_PATHS], Path(MANUAL_JSON_PATH)
else:
    from google.colab import files
    print("📂 Upload the PDF(s) AND the ground-truth .json together:")
    up = files.upload()
    for name, data in up.items():
        (bench_dir / name).write_bytes(data)
    pdf_paths = [bench_dir / n for n in up if n.lower().endswith(".pdf")]
    json_found = [bench_dir / n for n in up if n.lower().endswith(".json")]
    assert pdf_paths, "No PDF uploaded."
    assert json_found, "No ground-truth .json uploaded."
    json_path = json_found[0]

# The JSON has no document field, so one benchmark = one PDF. Pick which one.
target_pdf = next((p for p in pdf_paths if TARGET_PDF_HINT and TARGET_PDF_HINT.lower() in p.name.lower()), pdf_paths[0])
if len(pdf_paths) > 1:
    print(f"ℹ️ {len(pdf_paths)} PDFs uploaded - benchmarking against: {target_pdf.name} (set TARGET_PDF_HINT to change)")

bench = json.loads(json_path.read_text(encoding="utf-8"))
if isinstance(bench, dict):                                   # tolerate {"questions": [...]}
    bench = next(v for v in bench.values() if isinstance(v, list))
bench = [q for q in bench if q.get("question") and q.get("ground_truth_answer")]
if MAX_QUESTIONS:
    bench = bench[:MAX_QUESTIONS]
print(f"✅ {len(bench)} questions loaded from {json_path.name}")

# ---------------- 2) ingest + index the PDF (skipped if it is already the active one) ----------------
if globals().get("DOC_NAME") != target_pdf.name:
    print(f"📄 Ingesting {target_pdf.name} ...")
    PDF_PATH, DOC_NAME = target_pdf, target_pdf.name
    page_records, ocr_diags = extract_pages(PDF_PATH)
    docling_blocks = {}
    if CFG.use_docling:
        try:
            docling_blocks = run_docling(PDF_PATH)
        except Exception as e:
            print(f"⚠️ Docling failed ({type(e).__name__}) - using plain-text blocks.")
    parents, children = build_chunks(page_records)
    retriever = HybridRetriever(children, parents)       # the agent and reranker read this global
    DOC_OUTLINE, N_PAGES = build_outline(page_records), len(page_records)
else:
    print(f"♻️ {DOC_NAME} is already ingested - reusing the existing index.")
print(f"   {N_PAGES} pages | {len(children)} chunks indexed")

# ---------------- 3) scoring helpers ----------------
def parse_pages(loc):
    # "p. 1" | "pp. 1, 5–6" | "p. 4, Table 1" | "p. 1; p. 5"  ->  {1, 5, 6} etc.
    loc = re.sub(r"(?i)(table|figure|fig\.?)\s*\d+", " ", str(loc))     # drop "Table 1" / "Figure 3"
    pages = set()
    for a, b in re.findall(r"(\d+)\s*[–—-]\s*(\d+)", loc):
        pages.update(range(int(a), int(b) + 1))
    loc = re.sub(r"\d+\s*[–—-]\s*\d+", " ", loc)
    pages.update(int(n) for n in re.findall(r"\d+", loc))
    return pages


def _tokens(text):
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    return text.split()


def token_f1(pred, ref):
    p, r = Counter(_tokens(pred)), Counter(_tokens(ref))
    overlap = sum((p & r).values())
    if not overlap:
        return 0.0
    prec, rec = overlap / sum(p.values()), overlap / sum(r.values())
    return 2 * prec * rec / (prec + rec)


JUDGE_SYSTEM = """You grade a candidate answer against a reference answer for a question about a document.
- "correct": every key fact in the reference is present in the candidate and nothing contradicts it (extra detail is fine; wording may differ).
- "partial": some key facts are present but others are missing or vague.
- "incorrect": key facts are wrong or missing, or the candidate says the document does not contain the answer.
Reply with JSON only: {"verdict": "correct|partial|incorrect", "reason": "one short sentence"}"""


def judge(question, reference, candidate):
    raw = llm([{"role": "system", "content": JUDGE_SYSTEM},
               {"role": "user", "content": f"Question: {question}\n\nReference answer: {reference}\n\nCandidate answer: {candidate}"}],
              json_mode=True, max_tokens=200)
    out = parse_json(raw) or {}
    verdict = out.get("verdict", "incorrect")
    return {"correct": 1.0, "partial": 0.5}.get(verdict, 0.0), verdict, str(out.get("reason", ""))[:200]


# ---------------- 4) run the benchmark ----------------
rows = []
print(f"\n🚀 Running {len(bench)} questions ...\n")
for i, item in enumerate(bench, 1):
    gold = parse_pages(item.get("evidence_location", ""))
    row = {"id": item.get("id", f"Q{i:03d}"), "category": item.get("category", "n/a"),
           "question": item["question"], "reference": item["ground_truth_answer"],
           "gold_pages": sorted(gold)}
    try:
        res = rag.ask(item["question"], verbose=False)
        evidence = {s["page"] for s in res["sources"]}
        cited = set(res["cited_pages"])
        row.update(
            answer=res["answer"], operation=res["plan"]["operation"], filters=res["plan"]["filters"],
            sufficient=res["sufficient"], extra_retrievals=res["extra_retrievals"], seconds=res["seconds"],
            evidence_pages=sorted(evidence), cited_pages=sorted(cited),
            retrieval_hit=float(bool(gold & evidence)) if gold else None,
            retrieval_recall=len(gold & evidence) / len(gold) if gold else None,
            citation_hit=float(bool(gold & cited)) if gold else None,
            citation_precision=(len(gold & cited) / len(cited)) if (gold and cited) else None,
            no_citation=float(not cited),
            token_f1=token_f1(res["answer"], item["ground_truth_answer"]),
        )
        if USE_LLM_JUDGE:
            row["judge_score"], row["judge_verdict"], row["judge_reason"] = judge(
                item["question"], item["ground_truth_answer"], res["answer"])
    except Exception as e:
        row.update(error=f"{type(e).__name__}: {e}")
    rows.append(row)
    flag = "ERR" if "error" in row else f"{row.get('judge_verdict', '-'):<9} hit={row.get('retrieval_hit')} cite={row.get('citation_hit')}"
    print(f"[{i:>2}/{len(bench)}] {row['id']} {row['category']:<15} {flag}  ({row.get('seconds', 0)}s)")
    if PAUSE_SECONDS:
        time.sleep(PAUSE_SECONDS)

# ---------------- 5) summarise ----------------
df = pd.DataFrame(rows)
if "error" not in df.columns:
    df["error"] = None
metric_cols = [c for c in ["judge_score", "token_f1", "retrieval_hit", "retrieval_recall",
                           "citation_hit", "citation_precision", "no_citation", "seconds"] if c in df.columns]
ok = df[df["error"].isna()].copy()
for c in metric_cols:
    ok[c] = pd.to_numeric(ok[c], errors="coerce")        # None -> NaN so means skip them cleanly

print("\n" + "=" * 70 + "\nOVERALL\n" + "=" * 70)
print(ok[metric_cols].mean().round(3).to_string())
print(f"\nquestions answered: {len(ok)}/{len(df)} | errors: {int(df['error'].notna().sum())}")
print(f"extra retrieval triggered: {int((ok['extra_retrievals'] > 0).sum())} | judged insufficient at end: {int((~ok['sufficient'].astype(bool)).sum())}")

print("\n" + "=" * 70 + "\nBY CATEGORY\n" + "=" * 70)
print(ok.groupby("category")[metric_cols].mean().round(3).assign(n=ok.groupby("category").size()).to_string())

print("\n" + "=" * 70 + "\nBY CONTROLLER OPERATION\n" + "=" * 70)
print(ok.groupby("operation")[metric_cols].mean().round(3).assign(n=ok.groupby("operation").size()).to_string())

# Problem questions first, so you know where to look.
weak = pd.Series(False, index=ok.index)
if "judge_score" in ok.columns:
    weak |= ok["judge_score"] < 1.0
weak |= ok["retrieval_hit"] == 0
bad = ok[weak]
if len(bad):
    print("\n" + "=" * 70 + f"\nQUESTIONS TO INSPECT ({len(bad)}, showing up to 15)\n" + "=" * 70)
    for _, r in bad.head(15).iterrows():
        print(f"{r['id']} [{r['category']}] {r['question']}")
        print(f"    gold pages {r['gold_pages']} | evidence {r['evidence_pages']} | cited {r['cited_pages']} | op={r['operation']}")
        print(f"    ref : {r['reference'][:110]}")
        print(f"    got : {r['answer'][:110].replace(chr(10), ' ')}")
        if isinstance(r.get("judge_reason"), str) and r["judge_reason"]:
            print(f"    judge: {r['judge_verdict']} - {r['judge_reason']}")

# ---------------- 6) save + download ----------------
df.to_csv(bench_dir / "benchmark_results.csv", index=False)
(bench_dir / "benchmark_results.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
(bench_dir / "benchmark_summary.json").write_text(json.dumps({
    "document": DOC_NAME, "n_questions": len(df), "n_errors": int(df["error"].notna().sum()),
    "llm": LLM_LABEL, "overall": ok[metric_cols].mean().round(4).to_dict(),
    "by_category": ok.groupby("category")[metric_cols].mean().round(4).to_dict("index"),
}, indent=2), encoding="utf-8")

bench_zip = WORK / "benchmark_results.zip"
with zipfile.ZipFile(bench_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("benchmark_results.csv", "benchmark_results.json", "benchmark_summary.json"):
        z.write(bench_dir / f, f)
print(f"\n💾 Saved results in {bench_dir}")
try:
    from google.colab import files
    files.download(str(bench_zip))
except Exception as e:
    print(f"(Auto-download skipped: {e}) - zip is at {bench_zip}")